# 02 Cleaning & Enrichment

Clean the raw data according to project requirements, engineer new features, and export processed files to `data/processed/`.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / 'data/raw/Bookings.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
# Load data
df = pd.read_csv(RAW_PATH)
print(f"Initial shape: {df.shape}")

Initial shape: (103024, 21)


In [15]:
# 1. Drop Junk Columns
df = df.drop(columns=['Vehicle Images'], errors='ignore')
df = df.dropna(axis=1, how='all')

In [16]:
# 2. Fix Datetime
if 'Time' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'].astype(str), format='mixed', errors='coerce')
    df = df.drop(columns=['Time'])
else:
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')

df['Hour'] = df['Date'].dt.hour
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()

def get_time_of_day(hour):
    if pd.isna(hour): return np.nan
    if 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

df['Time_of_Day'] = df['Hour'].apply(get_time_of_day)

In [17]:
# 3. Standardize Categorical Columns
str_cols = df.select_dtypes(include=['object']).columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

for col in ['Pickup_Location', 'Drop_Location', 'Vehicle_Type', 'Booking_Status']:
    if col in df.columns:
        df[col] = df[col].str.title()

/var/folders/0r/pnxgc_2j22x0nhgs_tt4gcg00000gn/T/ipykernel_82897/3498843909.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include=['object']).columns


In [18]:
# 4. Remove Duplicates
df = df.drop_duplicates(subset=['Booking_ID'])

In [19]:
# 5. Validate Numeric Columns
for col in ['Ride_Distance', 'Booking_Value']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        anomalies = df[df[col] < 0]
        if len(anomalies) > 0:
            print(f"Warning: Found {len(anomalies)} negative values in {col}. Dropping them.")
            df = df[df[col] >= 0]
        else:
            print(f"{col} has no negative values.")

Ride_Distance has no negative values.
Booking_Value has no negative values.


In [20]:
# 6. Add Useful Engineered Columns
df['Is_Peak_Hour'] = df['Hour'].isin([7, 8, 9, 10, 17, 18, 19, 20, 21])
df['Is_Weekend'] = df['Day_of_Week'].isin(['Saturday', 'Sunday'])

if 'Booking_Value' in df.columns and 'Ride_Distance' in df.columns:
    df['Booking_Value_per_KM'] = np.where(df['Ride_Distance'] > 0, df['Booking_Value'] / df['Ride_Distance'], np.nan)

In [21]:
# 7. Split into Two Dataframes
success_df = df[df['Booking_Status'] == 'Success'].copy()
cancelled_df = df[df['Booking_Status'] != 'Success'].copy()

In [22]:
### 8. Clean success_df ###
for col in ['Driver_Ratings', 'Customer_Rating']:
    if col in success_df.columns:
        success_df[col] = pd.to_numeric(success_df[col], errors='coerce')
        success_df[col] = success_df[col].fillna(success_df[col].median())

success_df = success_df.dropna(subset=['V_TAT', 'C_TAT', 'Payment_Method'])

outlier_cols = ['V_TAT', 'C_TAT', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating']
for col in outlier_cols:
    if col in success_df.columns:
        success_df[col] = pd.to_numeric(success_df[col], errors='coerce')
        Q1 = success_df[col].quantile(0.25)
        Q3 = success_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        success_df = success_df[(success_df[col] >= lower_bound) & (success_df[col] <= upper_bound)]

for col in ['Driver_Ratings', 'Customer_Rating']:
    if col in success_df.columns:
        success_df[col] = success_df[col].fillna(success_df[col].mean())

cols_to_drop_success = ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides', 'Incomplete_Rides_Reason']
success_df = success_df.drop(columns=[col for col in cols_to_drop_success if col in success_df.columns])

In [23]:
### 9. Clean cancelled_df ###
cancel_fill_cols = ['Canceled_Rides_by_Customer', 'Canceled_Rides_by_Driver', 'Incomplete_Rides', 'Incomplete_Rides_Reason']
for col in cancel_fill_cols:
    if col in cancelled_df.columns:
        cancelled_df[col] = cancelled_df[col].fillna('N/A')

In [24]:
# 10. Export
success_df.to_csv(PROCESSED_DIR / 'cleaned_dataset.csv', index=False)
cancelled_df.to_csv(PROCESSED_DIR / 'cancelled_dataset.csv', index=False)

In [25]:
# 11. Final Verification
print("=== Success DataFrame ===")
print(f"Shape: {success_df.shape}")
print(f"Columns: {list(success_df.columns)}")
print(f"\nNull Counts:\n{success_df.isnull().sum()}\n")

print("=== Cancelled DataFrame ===")
print(f"Shape: {cancelled_df.shape}")
print(f"Columns: {list(cancelled_df.columns)}")
print(f"\nNull Counts:\n{cancelled_df.isnull().sum()}\n")

print("=== Value Counts (Original df) ===")
for col in ['Vehicle_Type', 'Booking_Status', 'Payment_Method', 'Time_of_Day']:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts())

=== Success DataFrame ===
Shape: (63967, 21)
Columns: ['Date', 'Booking_ID', 'Booking_Status', 'Customer_ID', 'Vehicle_Type', 'Pickup_Location', 'Drop_Location', 'V_TAT', 'C_TAT', 'Booking_Value', 'Payment_Method', 'Ride_Distance', 'Driver_Ratings', 'Customer_Rating', 'Hour', 'Day_of_Week', 'Month', 'Time_of_Day', 'Is_Peak_Hour', 'Is_Weekend', 'Booking_Value_per_KM']

Null Counts:
Date                    0
Booking_ID              0
Booking_Status          0
Customer_ID             0
Vehicle_Type            0
Pickup_Location         0
Drop_Location           0
V_TAT                   0
C_TAT                   0
Booking_Value           0
Payment_Method          0
Ride_Distance           0
Driver_Ratings          0
Customer_Rating         0
Hour                    0
Day_of_Week             0
Month                   0
Time_of_Day             0
Is_Peak_Hour            0
Is_Weekend              0
Booking_Value_per_KM    0
dtype: int64

=== Cancelled DataFrame ===
Shape: (39057, 25)
Columns: 